# 20 - RAG Evaluation

## 1. Setup paket

In [1]:
import subprocess, sys
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "pandas==2.2.3", "requests>=2.32,<3",
        "openai>=1.50,<2", "ragas>=0.4,<0.5",
        "langchain-community>=0.3,<0.4.2", "instructor>=1.3,<1.7" ])
    print("Selesai. RESTART KERNEL, lalu set INSTALL_DEPENDENCIES=False sebelum lanjut.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.2/324.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.109.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


Selesai. RESTART KERNEL, lalu set INSTALL_DEPENDENCIES=False sebelum lanjut.


## 2. Muat konfigurasi, prediksi, dan gold chunk 

In [2]:
from pathlib import Path
import os, json, hashlib, time, shutil
from collections import Counter
import pandas as pd
import numpy as np
import requests

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def read_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Aktifkan secret {name} pada Kaggle Add-ons > Secrets.")
    return value

RUN_NAME = "Lengkap"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT = WORK / "rag_results" / RUN_NAME
PRIOR_RUN_DIR = "/kaggle/input/notebooks/muhammadnabilul/19-generate-prediction-rag-k1-k5/rag_results/Lengkap"

shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)
for p in Path(PRIOR_RUN_DIR).rglob("*"):
    if p.is_file():
        target = OUT / p.relative_to(PRIOR_RUN_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, target)

manifest = json.loads((OUT / "manifest.json").read_text(encoding="utf-8"))
SCENARIOS = manifest.get("scenarios")

raw_records = [json.loads(p.read_text(encoding="utf-8")) for p in (OUT / "predictions").glob("*.json")]
id_counts = Counter(r.get("experiment_id") for r in raw_records if r.get("experiment_id"))
if not id_counts:
    raise ValueError(f"Tidak ada experiment_id di predictions/ ({len(raw_records)} file ditemukan). PRIOR_RUN_DIR salah.")
EXPERIMENT_ID, _ = id_counts.most_common(1)[0]
if len(id_counts) > 1:
    print("PERINGATAN >1 experiment_id ditemukan:", dict(id_counts))

records = [r for r in raw_records if r.get("experiment_id") == EXPERIMENT_ID and r.get("status") == "ok"]
pred = pd.DataFrame(records)

gold = pd.read_json(OUT / "gold_chunks.jsonl", lines=True)
gold_by_id = {r.record_id: set(r.gold_chunk_ids) for r in gold.itertuples()}

print(len(raw_records), "total file,", len(pred), "prediksi valid dimuat, dari", pred.record_id.nunique(), "pertanyaan unik")
print(pred.groupby(["condition", "protocol"]).size())

1246 total file, 1246 prediksi valid dimuat, dari 89 pertanyaan unik
condition  protocol  
K1         end_to_end    178
K2         controlled    178
           end_to_end    178
K3         end_to_end    178
K4         controlled    178
           end_to_end    178
K5         controlled     89
           end_to_end     89
dtype: int64


In [3]:
import json
from pathlib import Path
from collections import Counter

FIXED_GOLD = Path("/kaggle/input/notebooks/muhammadnabilul/18-persiapan-evaluasi-rag/rag_results/Penuh/gold_chunks.jsonl")
rows = [json.loads(l) for l in FIXED_GOLD.read_text(encoding="utf-8").splitlines() if l.strip()]

n_chunks = [len(r["gold_chunk_ids"]) for r in rows]
has_delimiter_bug = any("|||" in cid for r in rows for cid in r["gold_chunk_ids"])
fallback_rate = sum(r.get("fallback_used", False) for r in rows) / len(rows)

print("n records:", len(rows))
print("avg n_context_chunks:", sum(n_chunks)/len(n_chunks))
print("distribusi n_chunks:", Counter(n_chunks))
print("ada bug delimiter '|||' ?", has_delimiter_bug)
print("fallback_rate:", fallback_rate)

n records: 89
avg n_context_chunks: 1.898876404494382
distribusi n_chunks: Counter({1: 62, 2: 15, 3: 4, 8: 3, 6: 2, 5: 1, 10: 1, 14: 1})
ada bug delimiter '|||' ? False
fallback_rate: 0.06741573033707865


In [4]:
import shutil
shutil.copy2(FIXED_GOLD, OUT / "gold_chunks.jsonl")

gold = pd.read_json(OUT / "gold_chunks.jsonl", lines=True)
gold_by_id = {r.record_id: set(r.gold_chunk_ids) for r in gold.itertuples()}
print("gold_chunks.jsonl diganti, jumlah record:", len(gold))

gold_chunks.jsonl diganti, jumlah record: 89


## 3. Hit@5, Recall@5, MRR - deterministik, seluruh prediksi RAG

In [5]:
K = 5

# buang suffix ::faiss=N sebelum dibandingkan ke gold_chunk_ids (yang berupa chunk_id polos)
def strip_faiss_suffix(eval_id):
    return eval_id.split("::faiss=")[0]

# Hit@k: 1 kalau minimal satu gold chunk ada di top-k hasil retrieval, else 0
def hit_at_k(retrieved_ids, gold_ids, k=K):
    top_k = {strip_faiss_suffix(i) for i in retrieved_ids[:k]}
    return 1.0 if top_k & gold_ids else 0.0

# Recall@k: proporsi gold chunk yang berhasil ditemukan di top-k
def recall_at_k(retrieved_ids, gold_ids, k=K):
    if not gold_ids:
        return None
    top_k = {strip_faiss_suffix(i) for i in retrieved_ids[:k]}
    return len(top_k & gold_ids) / len(gold_ids)

# MRR: 1/rank kemunculan pertama gold chunk di ranking penuh (bukan cuma top-k)
def reciprocal_rank(ranked_ids, gold_ids):
    for rank, rid in enumerate(ranked_ids, 1):
        if strip_faiss_suffix(rid) in gold_ids:
            return 1.0 / rank
    return 0.0

rag_pred = pred[pred.use_rag].copy()
rag_pred["gold_ids"] = rag_pred.record_id.map(gold_by_id)
rag_pred["hit"] = rag_pred.apply(lambda r: hit_at_k(r.retrieved_ids, r.gold_ids), axis=1)
rag_pred["recall"] = rag_pred.apply(lambda r: recall_at_k(r.retrieved_ids, r.gold_ids), axis=1)
rag_pred["rr"] = rag_pred.apply(
    lambda r: reciprocal_rank(r.retrieval.get("ranked_ids", r.retrieved_ids), r.gold_ids), axis=1)

# controlled sama untuk semua model (retrieval dikunci REFERENCE_MODEL) -> laporkan 1 baris saja
retrieval_metrics = (
    rag_pred.groupby(["condition", "architecture", "protocol"])
    .agg(hit_at_5=("hit", "mean"), recall_at_5=("recall", "mean"),
         mrr=("rr", "mean"), n=("hit", "size"))
    .reset_index()
)
controlled_dedup = (retrieval_metrics.protocol.eq("controlled") &
                     retrieval_metrics.duplicated(subset=["protocol"], keep="first"))
retrieval_metrics_report = retrieval_metrics[~controlled_dedup].copy()
retrieval_metrics_report.loc[retrieval_metrics_report.protocol.eq("controlled"),
    ["condition", "architecture"]] = ["K2/K4/K5", "semua (retrieval dikunci)"]

print(retrieval_metrics_report.to_string(index=False))
retrieval_metrics.to_csv(OUT / "retrieval_metrics_detail.csv", index=False)
retrieval_metrics_report.to_csv(OUT / "retrieval_metrics_report.csv", index=False)

condition              architecture   protocol  hit_at_5  recall_at_5      mrr  n
 K2/K4/K5 semua (retrieval dikunci) controlled  0.595506     0.530150 0.399854 89
       K2                    gemma2 end_to_end  0.561798     0.488483 0.378283 89
       K2                  llama3.2 end_to_end  0.584270     0.518914 0.388899 89
       K4                    gemma2 end_to_end  0.606742     0.509283 0.401548 89
       K4                  llama3.2 end_to_end  0.595506     0.513497 0.402331 89
       K5                  deepseek end_to_end  0.584270     0.495840 0.401006 89


In [6]:
subtask_hit = rag_pred.groupby("subtask_type")["hit"].agg(["mean", "count"]).sort_values("mean")
print(subtask_hit)

                              mean  count
subtask_type                             
document_overview         0.200000     50
definition_clarification  0.300000    140
source_navigation         0.500000    140
methodology_explanation   0.583333    120
causal_reasoning          0.612500     80
factual_retrieval         0.666667    120
comparative_analysis      0.737500     80
trend_interpretation      0.900000     80
multi_source_synthesis    0.925000     80


## 4. Metrik RAGAS - Faithfulness & ResponseRelevancy (library resmi), Context Relevance (Nvidia, library resmi)

In [7]:
import importlib.metadata as m
print("instructor:", m.version("instructor"))
print("ragas:", m.version("ragas"))

import instructor
print("punya patch()?", hasattr(instructor, "patch"))

instructor: 1.6.4
ragas: 0.4.3
punya patch()? True


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


In [8]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRelevance

RAGAS_JUDGE_MODEL = "gpt-4o-mini"
RAGAS_EMBED_MODEL = "text-embedding-3-small"
RUN_RAGAS = True

from openai import AsyncOpenAI
_client = AsyncOpenAI(api_key=read_secret("API_OpenAI", required=True))
_llm = llm_factory(RAGAS_JUDGE_MODEL, client=_client, max_tokens=4096)
_embeddings = OpenAIEmbeddings(client=_client, model=RAGAS_EMBED_MODEL)

faithfulness_scorer = Faithfulness(llm=_llm)
answer_relevancy_scorer = AnswerRelevancy(llm=_llm, embeddings=_embeddings)
context_relevance_scorer = ContextRelevance(llm=_llm)

PRIOR_RAGAS_CACHE = Path("/kaggle/input/datasets/muhammadnabilul/checkpoint-nb20/rag_results/Lengkap/ragas_cache")

CACHE_DIR = OUT / "ragas_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if PRIOR_RAGAS_CACHE.exists():
    n = 0
    for p in PRIOR_RAGAS_CACHE.glob("*.json"):
        shutil.copy2(p, CACHE_DIR / p.name)
        n += 1
    print(n, "file cache RAGAS di-restore dari checkpoint")
else:
    print("PRIOR_RAGAS_CACHE tidak ditemukan:", PRIOR_RAGAS_CACHE)

# cache hasil skor per (record_id, metric, model_key, condition, protocol) - hindari panggil ulang API
def cached_score(record_id, metric, model_key, condition, protocol, coro_fn):
    key = digest([record_id, metric, model_key, condition, protocol])
    path = CACHE_DIR / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))["value"]
    async def _run():
        result = await coro_fn()
        return result.value
    import asyncio
    value = asyncio.get_event_loop().run_until_complete(_run())
    write_json(path, {"value": value})
    return value

1961 file cache RAGAS di-restore dari checkpoint


## 5. Jalankan RAGAS ke seluruh prediksi (dengan cache dan guard biaya)

In [9]:
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [10]:
sample = pred.sample(10, random_state=42)
for _, row in sample.iterrows():
    print("="*80)
    print(f"record_id={row.record_id} | condition={row.condition} | model={row.model_key} | protocol={row.protocol}")
    print(f"subtask_type={getattr(row, 'subtask_type', 'N/A')} | use_rag={row.use_rag}")
    print(f"Q: {row.question}")
    print(f"A ({len(row.answer or '')} char): {row.answer[:400]}")
    if row.use_rag and row.contexts:
        print(f"Context (200 char pertama): {row.contexts[0][:200]}")

record_id=93dfbaa6fc49b718 | condition=K4 | model=gemma2-finetuned | protocol=controlled
subtask_type=source_navigation | use_rag=True
Q: Pada halaman atau bagian mana dari laporan ini data Indeks Khusus Penanganan Stunting untuk tahun 2021-2022 dapat ditemukan?
A (93 char): Data Indeks Khusus Penanganan Stunting untuk tahun 2021-2022 terdapat pada halaman 26 dan 27.
Context (200 char pertama): [1] Laporan Indeks Khusus Penanganan Stunting 2021–2022 | sec_025
26
Laporan Indeks Khusus Penanganan Stunting 2021-2022

Pada tahun 2021-2022, indeks pada dimensi kesehatan, gizi, perumahan, dan dime
record_id=09c468a22018f683 | condition=K1 | model=llama3.2-base | protocol=end_to_end
subtask_type=factual_retrieval | use_rag=False
Q: Apa yang dimaksud dengan Akreditasi Perpustakaan Khusus menurut dokumen?
A (163 char): Menurut dokumen, Akreditasi Perpustakaan Khusus adalah proses penilaian dan pengakuan keakuratan dan keabsahan koleksi, perpustakaan, dan layanan perpustakaan. [1]
record_id=d2af

In [11]:
import re, shutil

def strip_quotes_for_ragas(text):
    text = re.sub(r'\[\d+\]', '', text)
    text = text.replace('"', '').replace('"', '').replace('"', '')
    return text.strip()

def repetition_ratio(text):
    if not isinstance(text, str) or len(text) < 50:
        return 0.0
    compressed = re.sub(r'(.{1,15}?)\1{4,}', r'\1', text)
    return 1 - (len(compressed) / len(text))

DEGENERATE_THRESHOLD = 0.3
MIN_LEN_FOR_AR = 15  # panjang minimum (setelah stripping) agar AnswerRelevancy dihitung

# shutil.rmtree(CACHE_DIR, ignore_errors=True)
# CACHE_DIR.mkdir(parents=True, exist_ok=True)

def run_ragas_row(row):
    result = {"record_id": row.record_id, "model_key": row.model_key,
               "architecture": row.architecture, "condition": row.condition,
               "protocol": row.protocol}
    context = row.contexts[0] if row.contexts else ""
    answer = row.answer if isinstance(row.answer, str) else ""

    if not answer.strip():
        result.update(answer_relevancy=None, faithfulness=None, context_relevance=None,
                       ragas_status="empty_answer")
        return result
    if repetition_ratio(answer) > DEGENERATE_THRESHOLD:
        result.update(answer_relevancy=None, faithfulness=None, context_relevance=None,
                       ragas_status="degenerate_answer")
        return result

    answer_clean = strip_quotes_for_ragas(answer)

    if len(answer_clean) < MIN_LEN_FOR_AR:
        result["answer_relevancy"] = None
        ar_status = "too_short_for_ar"
    else:
        try:
            ar = cached_score(row.record_id, "answer_relevancy", row.model_key, row.condition, row.protocol,
                lambda: answer_relevancy_scorer.ascore(user_input=row.question, response=answer_clean))
            result["answer_relevancy"] = ar
            ar_status = "ok"
        except Exception as e:
            result["answer_relevancy"] = None
            ar_status = f"ar_error:{type(e).__name__}"

    if row.use_rag and context:
        try:
            fa = cached_score(row.record_id, "faithfulness", row.model_key, row.condition, row.protocol,
                lambda: faithfulness_scorer.ascore(user_input=row.question, response=answer,
                                                    retrieved_contexts=[context]))
            result["faithfulness"] = fa
            cr = cached_score(row.record_id, "context_relevance", row.model_key, row.condition, row.protocol,
                lambda: context_relevance_scorer.ascore(user_input=row.question, retrieved_contexts=[context]))
            result["context_relevance"] = cr
        except Exception as e:
            result["faithfulness"] = None
            result["context_relevance"] = None
            ar_status = (ar_status + f"|rag_error:{type(e).__name__}") if ar_status != "ok" else f"rag_error:{type(e).__name__}"
    else:
        result["faithfulness"] = None
        result["context_relevance"] = None

    result["ragas_status"] = ar_status
    return result

if RUN_RAGAS:
    ragas_rows = []
    for i, row in enumerate(pred.itertuples(), 1):
        r = run_ragas_row(row)
        ragas_rows.append(r)
        ar_val = r["answer_relevancy"]
        fa_val = r["faithfulness"]
        cr_val = r["context_relevance"]
        ar_str = f"{ar_val:.3f}" if ar_val is not None else "N/A"
        fa_str = f"{fa_val:.3f}" if fa_val is not None else "N/A"
        cr_str = f"{cr_val:.3f}" if cr_val is not None else "N/A"
        print(f"[{i}/{len(pred)}] {row.record_id} | {row.condition}/{row.model_key}/{row.protocol} "
              f"| AR={ar_str} | F={fa_str} | CR={cr_str} | status={r['ragas_status']}")
    ragas_df = pd.DataFrame(ragas_rows)
    ragas_df.to_csv(OUT / "ragas_raw.csv", index=False)
else:
    ragas_path = OUT / "ragas_raw.csv"
    ragas_df = pd.read_csv(ragas_path) if ragas_path.exists() else pd.DataFrame()
    print("RUN_RAGAS=False - memuat hasil tersimpan" if len(ragas_df) else
          "RUN_RAGAS=False dan belum ada hasil tersimpan - set True untuk menjalankan")

[1/1246] a06f8affce357a38 | K1/gemma2-base/end_to_end | AR=0.000 | F=N/A | CR=N/A | status=ok
[2/1246] 50f3d60f659cb1b9 | K4/llama3.2-finetuned/controlled | AR=0.588 | F=0.600 | CR=1.000 | status=ok
[3/1246] 08353c2b983b876a | K3/gemma2-finetuned/end_to_end | AR=0.546 | F=N/A | CR=N/A | status=ok
[4/1246] 2be7ac91584146a6 | K1/llama3.2-base/end_to_end | AR=0.000 | F=N/A | CR=N/A | status=ok
[5/1246] 7aa0ed4d7a7f7dda | K4/llama3.2-finetuned/controlled | AR=0.451 | F=1.000 | CR=1.000 | status=ok
[6/1246] 08603bf330142ac9 | K1/llama3.2-base/end_to_end | AR=0.000 | F=N/A | CR=N/A | status=ok
[7/1246] a7ecfde07e41aa25 | K4/gemma2-finetuned/controlled | AR=0.658 | F=0.750 | CR=1.000 | status=ok
[8/1246] a57bbfca1ad82c22 | K4/gemma2-finetuned/end_to_end | AR=0.686 | F=1.000 | CR=1.000 | status=ok
[9/1246] 7cad5ec3341f2102 | K1/llama3.2-base/end_to_end | AR=0.621 | F=N/A | CR=N/A | status=ok
[10/1246] a57bbfca1ad82c22 | K2/gemma2-base/controlled | AR=0.846 | F=1.000 | CR=1.000 | status=ok
[11/

In [12]:
print(ragas_df["ragas_status"].value_counts())

ragas_status
ok                                    1191
too_short_for_ar                        49
degenerate_answer                        5
rag_error:InstructorRetryException       1
Name: count, dtype: int64


## 6. Ringkasan RAGAS per kondisi dan selisih berpasangan

In [13]:
if len(ragas_df):
    ragas_summary = (
        ragas_df.groupby(["condition", "architecture", "protocol"])
        .agg(faithfulness=("faithfulness", "mean"), answer_relevancy=("answer_relevancy", "mean"),
             context_relevance=("context_relevance", "mean"), n=("record_id", "size"))
        .reset_index()
    )
    ragas_summary["n_answer_relevancy"] = (
        ragas_df.groupby(["condition", "architecture", "protocol"])["answer_relevancy"]
        .apply(lambda s: s.notna().sum()).values
    )
    print(ragas_summary.to_string(index=False))
    ragas_summary.to_csv(OUT / "ragas_summary.csv", index=False)

    def paired_diff(df, cond_a, cond_b, arch, protocol_a="end_to_end", protocol_b="end_to_end", metrics=None):
        metrics = metrics or ["faithfulness", "answer_relevancy", "context_relevance"]
        a = df[(df.condition == cond_a) & (df.architecture == arch) & (df.protocol == protocol_a)]
        b = df[(df.condition == cond_b) & (df.architecture == arch) & (df.protocol == protocol_b)]
        merged = a.merge(b, on="record_id", suffixes=("_a", "_b"))
        out = {}
        for metric in metrics:
            if f"{metric}_a" in merged and f"{metric}_b" in merged:
                out[metric] = (merged[f"{metric}_a"] - merged[f"{metric}_b"]).mean()
        return out

    diffs = []
    for arch in ["llama3.2", "gemma2"]:
        diffs.append({"comparison": f"K2-K1 ({arch}, efek RAG di base)",
                       **paired_diff(ragas_df, "K2", "K1", arch, metrics=["answer_relevancy"])})
        diffs.append({"comparison": f"K4-K3 ({arch}, efek RAG di fine-tuned)",
                       **paired_diff(ragas_df, "K4", "K3", arch, metrics=["answer_relevancy"])})
        diffs.append({"comparison": f"K4-K2 ({arch}, efek fine-tuning, controlled)",
                       **paired_diff(ragas_df, "K4", "K2", arch, protocol_a="controlled", protocol_b="controlled")})
    diffs_df = pd.DataFrame(diffs)
    print("\n", diffs_df.to_string(index=False))
    diffs_df.to_csv(OUT / "ragas_paired_differences.csv", index=False)

condition architecture   protocol  faithfulness  answer_relevancy  context_relevance  n  n_answer_relevancy
       K1       gemma2 end_to_end           NaN          0.358900                NaN 89                  74
       K1     llama3.2 end_to_end           NaN          0.316934                NaN 89                  89
       K2       gemma2 controlled      0.779093          0.574682           0.963483 89                  72
       K2       gemma2 end_to_end      0.810879          0.569866           0.949438 89                  72
       K2     llama3.2 controlled      0.795771          0.684764           0.957865 89                  89
       K2     llama3.2 end_to_end      0.759525          0.673227           0.963483 89                  89
       K3       gemma2 end_to_end           NaN          0.575942                NaN 89                  85
       K3     llama3.2 end_to_end           NaN          0.577945                NaN 89                  88
       K4       gemma2 contr